# Produce the training data for a detector

A detector belongs to the model it scores, so training one for **your** model starts by
recording how that model answers: the questions, its answers *with the token
log-probabilities behind them*, and a verdict on each answer.

This notebook produces all three, against any OpenAI-compatible endpoint.

1. **Draw the questions** — from a QA dataset with short answers, or your own pack.
2. **Answer them**, keeping `top_logprobs`. The distribution behind the answer is what the
   detector reads; the answer text is only used to judge it.
3. **Judge the answers** against their gold answers, with the model as judge.
4. **Fit and evaluate** a WEPR detector on the result, and save it.

**No GPU.** Every model call goes to the endpoint; everything this notebook runs locally is
bookkeeping and a logistic regression.

## What it writes

Three files, joined on `custom_id`:

| File | Contents |
|---|---|
| `questions.json` | The question pack: question, id, gold answer, aliases |
| `responses.jsonl` | The answers, with `top_logprobs` per token |
| `judgments.jsonl` | The judge's reply for each answer |

The two `.jsonl` files are the **OpenAI Batch output shape** — one JSON object per line,
each wrapping a chat completion under `custom_id`. That format is not this notebook's
invention and not its private convention: it is what the Batch API returns, so these files
are readable by anything that reads a batch, and can be produced by anything that writes
one. Keeping to it is why nothing downstream needs a conversion step.

Writing them out at all is the point. Generating and judging cost money and time; fitting
costs seconds. With the files on disk you can refit at a different `k`, on `epr` instead of
`wepr`, or against relabelled verdicts, without paying for any of it twice.

## Prerequisites

```bash
uv pip install "artefactual[adapters]" datasets
```

| Variable | Required | What it is |
|---|---|---|
| `OPENAI_BASE_URL` | yes | Any OpenAI-compatible endpoint returning `top_logprobs` |
| `OPENAI_API_KEY` | yes | Its key |
| `OPENAI_MODEL` | yes | The model being scored — the detector you train belongs to it, and the id has to be one your endpoint serves |

**The endpoint has to return at least `K` ranks per token**, which is the one requirement
worth checking before you start. OpenAI's own API accepts `top_logprobs` up to 20, and a
self-hosted vLLM up to its `--max-logprobs` (20 by default), so `K = 15` fits both.
Providers that cap lower, or omit `logprobs` entirely, are refused by name in step 2 rather
than silently training on narrower data.

**A detector belongs to the model it was trained on.** Its weights read that model's
confidence, so scoring a different model with them is not supported; retrain instead.

This notebook is not executed when the documentation is built, because it generates against
a live endpoint. The numbers you see are the ones your run produces.

In [ ]:
# Colab, or any other kernel that is not this repository's environment: install what this
# notebook needs.
# A checkout that ran `uv sync --extra adapters` has all of it already, so nothing
# below runs there.
#
# uv rather than pip, because pip is the slow half of the wait: installing this package
# into an empty environment measured 17 s under pip, against 3 s to pip-install uv plus 1 s
# for uv to do the same work. On Colab, where most of the dependency tree is already
# present, the gap is smaller.
#
# Plain Python rather than the `!pip` and `%pip` magics, so the cell stays valid Python:
# the tests that run these notebooks compile the code cells, and so do the linters.
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path


def missing(distribution):
    """Whether `distribution` is installed in the interpreter running this kernel.

    Asked of the installed distribution rather than of an import, because a bare directory
    named `artefactual` -- which is what cloning this repository beside the notebook leaves
    behind -- is an empty namespace package: `find_spec` finds it and `import artefactual`
    succeeds, so both would report the package present and skip the install. Only the
    metadata distinguishes a directory from a package.
    """
    try:
        importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return True
    return False


def shadowed(name):
    """Whether a directory beside the notebook hides an installed package of that name.

    The working directory comes first on `sys.path`, so such a directory wins over
    anything installed and the import fails on a submodule, several cells from the cause.
    """
    return Path(name).is_dir() and not Path(name, "__init__.py").exists()


def install(*packages):
    """Install into the interpreter running this kernel, showing what went wrong if it does.

    `-q` and no captured output is how an install failure becomes a bare
    `CalledProcessError` with the resolver's explanation nowhere on screen.
    """
    bootstrap = subprocess.run([sys.executable, "-m", "pip", "install", "-qU", "uv"], capture_output=True, text=True)
    resolve = [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, "-q", *packages]
    done = bootstrap if bootstrap.returncode else subprocess.run(resolve, capture_output=True, text=True)
    if done.returncode:
        print(done.stderr or done.stdout)
        done.check_returncode()
    # The kernel started before these files existed, so the import machinery has a cached
    # listing of a directory that did not contain them.
    importlib.invalidate_caches()


if missing("artefactual") or missing("openai") or missing("datasets"):
    install("artefactual[adapters]", "datasets")

assert not shadowed("artefactual"), (
    "a directory named 'artefactual' beside this notebook is hiding the installed "
    "package; rename it, or run this notebook from somewhere else"
)

In [ ]:
import contextlib
import json
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# No default: a model id only means something to the endpoint serving it, and a wrong one
# fails on every request in step 2 rather than here.
MODEL = os.environ["OPENAI_MODEL"]

# Ranks kept per token. Part of the feature definition, not a batch size: WEPR fits one
# coefficient per rank, so a detector is only ever used at the k it was fitted at. Every
# published detector uses 15.
K = 15
# The API caps it at 20, and an endpoint asked for more rejects every request in step 2 --
# which arrives as "no answers were generated" three cells later, pointing at the wrong
# thing.
assert 1 <= K <= 20, "top_logprobs must be between 1 and 20"

# Start small. This is 2 requests per question -- one to answer, one to judge -- so 25
# questions is 50 requests and a minute or two. It is a smoke test, not a training run: at
# this size a capable model may hallucinate two or three times, which is too few to split
# and too few to score. Raise it once a run has come back clean; a few hundred is where the
# numbers start to mean something.
N_QUESTIONS = 25
SEED = 42
WORKERS = 8

QUESTIONS = Path("questions.json")
RESPONSES = Path("responses.jsonl")
JUDGMENTS = Path("judgments.jsonl")

## Step 1 — build the question pack

Four fields per question. `question_id` is the one that travels: it becomes `custom_id` on
the answers and the verdicts, and that is what every later join pairs on.

TriviaQA's closed-book configuration (`rc.nocontext`) carries all four, so the mapping is a
rename. **Any short-form QA set works** — only these column names change. Two properties
decide whether one is usable, and neither is about its columns: answers must be **short
enough for a judge to grade** against `short_answer`, and the model must get **enough of
them wrong** that both classes appear. A model that answers everything correctly leaves the
fit nothing to learn from.

Shuffled before slicing, because splits arrive grouped by source and the head of one is a
narrower sample than the same count drawn at random.

Written to disk rather than kept in memory, so a rerun of the later steps answers the
same questions -- and so a pack you already have can be dropped in instead of this cell.

In [ ]:
from datasets import load_dataset

rows = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")

questions = [
    {
        "question": row["question"],
        "question_id": row["question_id"],
        "short_answer": row["answer"]["value"],
        # `value` goes in its own field, so the aliases that merely restate it are dropped
        # rather than repeated to the judge.
        "answer_aliases": [a for a in row["answer"]["aliases"] if a.casefold() != row["answer"]["value"].casefold()],
    }
    for row in rows.shuffle(seed=SEED).select(range(N_QUESTIONS))
]

# Duplicate ids would collapse in the join, pairing an answer with another question's gold
# answer -- a wrong label rather than an error, so it is refused here.
assert len({q["question_id"] for q in questions}) == len(questions), "question_id is not unique"

QUESTIONS.write_text(json.dumps(questions, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"wrote {QUESTIONS}, {len(questions)} questions")
print(json.dumps(questions[0], indent=2)[:400])

## Step 2 — generate the answers, keeping the log-probabilities

`logprobs=True` and `top_logprobs=K` are what make a response scoreable at all: the
detector reads the token distribution behind the answer, not the answer. A response
generated without them is a valid completion carrying nothing to score, and one generated
with fewer than `K` ranks is refused when it reaches the parser rather than zero-filled —
the missing ranks are unfetched rather than absent, so filling them with zeros would score
the answer as more confident than it was. Generating *wider* than `K` is safe; surplus
ranks are dropped.

Prompt and sampling follow the paper (§4.1.2): non-greedy at `T = 1.0`, `top_p = 1.0`.
The paper also sets `top_k = 50`, which is not an OpenAI parameter — so this samples at
whatever the endpoint's default is, and the distribution is not quite the paper's. It also
sends `max_completion_tokens`, which some OpenAI-compatible servers still only accept as
`max_tokens`; if every request fails, that is the first thing to check.
Non-greedy is the point — the method measures hesitation in the raw distribution.

Requests are threaded because the round trips, not the fitting, are what make this slow. A
failure returns `None` and is written out as an error row, exactly as a batch job records
one, so the count is visible rather than silently missing.

In [ ]:
from openai import OpenAI, OpenAIError

# `max_retries` above the SDK's default of 2: this fires N requests at once, and a burst
# of 429s that exhausts the retries becomes a thinner dataset rather than an error.
client = OpenAI(max_retries=6)  # reads OPENAI_BASE_URL and OPENAI_API_KEY

# Which endpoint this is actually talking to. Unset, OPENAI_BASE_URL silently means
# api.openai.com, and a self-hosted run then fails N times with an authentication error.
print(f"endpoint: {client.base_url}")

GENERATE = """You are a useful assistant that help finding short and precise answers for a given query or question.
            Please keep your output AS SHORT AND CONCISE AS POSSIBLE.
            Here is the query :
            {query}
            """


def generate(question):
    try:
        return client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": GENERATE.format(query=question["question"])}],
            logprobs=True,
            top_logprobs=K,
            temperature=1.0,
            top_p=1.0,
            max_completion_tokens=200,
        )
    except OpenAIError as error:
        # Every failure this call can produce -- transport, timeout, rate limit, a
        # rejected request -- is an OpenAIError, and one of them should not cost the
        # run. Anything else is a bug in the code above and should not be caught here.
        return error


with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    generated = list(pool.map(generate, questions))  # `map` preserves order

# The OpenAI Batch output shape: one line per request, `custom_id` carrying the question id
# and the ChatCompletion in `response.body`. Failures become error rows rather than gaps.
with RESPONSES.open("w", encoding="utf-8") as out:
    for question, result in zip(questions, generated):
        failed = isinstance(result, Exception)
        out.write(
            json.dumps(
                {
                    "id": f"chatcmpl-{question['question_id']}",
                    "custom_id": question["question_id"],
                    "response": None if failed else {"status_code": 200, "body": result.model_dump()},
                    # The class name, not the provider's text: an authentication error
                    # quotes the key it rejected, and this file is one you hand onward.
                    # The full message is printed below, where it stays in the session.
                    "error": {"message": type(result).__name__} if failed else None,
                },
                # ASCII-escaped, unlike questions.json: every reader of these files splits
                # them with `splitlines()`, which breaks on U+2028, U+2029 and U+0085 --
                # characters JSON does not require escaping and a model can emit.
                ensure_ascii=True,
            )
            + "\n"
        )

ok = [(q, r) for q, r in zip(questions, generated) if not isinstance(r, Exception)]
print(f"wrote {RESPONSES}, {len(ok)}/{len(generated)} generated")
for question, result in zip(questions, generated):
    if isinstance(result, Exception):
        print(f"  failed: {question['question_id']}: {result}")

In [ ]:
# Nothing came back at all -- almost always OPENAI_BASE_URL, the key, or a model name the
# endpoint does not serve. Checked before indexing, because the errors above say what
# happened and a bare IndexError here would not.
assert ok, (
    "no answers were generated. Check OPENAI_BASE_URL, OPENAI_API_KEY and OPENAI_MODEL, "
    "and read the per-request errors above: an endpoint that rejects `top_logprobs`, "
    "`temperature` or `max_completion_tokens` fails every request the same way."
)

# The cheapest place to notice an endpoint that ignored `top_logprobs`: it returns a
# perfectly valid completion carrying nothing to score. `logprobs` is checked before
# `.content` because that is the shape the failure actually takes -- reaching straight for
# `.content` raises an AttributeError that names nothing useful.
# `.content` is optional inside `logprobs` as well, and a provider that returns the
# object with nothing in it fails the same way for the reader.
missing = [q["question_id"] for q, r in ok if not (r.choices[0].logprobs and r.choices[0].logprobs.content)]
assert not missing, (
    f"{len(missing)} response(s) carry no logprobs at all, e.g. {missing[:3]}. "
    f"The endpoint accepted `logprobs=True` and ignored it; it cannot be used at any k."
)

# Every token, not just the first -- a response whose later tokens are narrower would pass
# a first-token check and fail inside `fit`.
widths = [len(t.top_logprobs) for _, r in ok for t in (r.choices[0].logprobs.content or [])]
assert widths and min(widths) >= K, (
    f"endpoint returned {min(widths) if widths else 0} ranks per token, need {K}; "
    f"raise top_logprobs, or lower K and fit at that rank count"
)

print(f"{min(widths)}-{max(widths)} ranks per token across {len(ok)} answers")
print(f"{ok[0][0]['question']}\n  -> {ok[0][1].choices[0].message.content.strip()}")

## Step 3 — judge the answers

The judge prompt is the paper's, inlined below in full. It shows the judge the question,
the gold answer, its aliases and the generated answer, and asks for
`{"judgment": true|false, "explanation": "..."}` -- a semantic comparison rather than a
string match, so an answer that is right but worded differently is accepted.

**The judge answers the opposite question to the one we record.** Its `judgment: true`
means the answer was **correct**; the file we write states `hallucination`, the positive
class the detector predicts. The conversion is `hallucination = not judgment`, done once,
at the point of writing.

Rendering is by literal replacement rather than `str.format`: the prompt ends with a JSON
example whose braces `format` would read as fields, and a question containing a brace would
corrupt everything after it.

Grading is deterministic — `temperature=0` — and a verdict that will not parse is dropped
with its answer rather than guessed at.

In [ ]:
JUDGE = """You are an expert evaluator tasked with determining if two answers convey compatible information. Your task is to make a binary True/False judgment on whether the answers are SEMANTICALLY COMPATIBLE.

Query:
{{query}}

Expected Answer:
{{expected_answer}}
{{aliases_block}}
Generated Answer:
{{generated_answer}}

CRITICAL INSTRUCTIONS:
1. FIRST, perform a simple VERBATIM TEXT COMPARISON:
   - If the generated answer is IDENTICAL (exact same text) to EITHER the expected answer OR ANY of the answer aliases, your judgment MUST be TRUE
   - If not identical to any of them, proceed to semantic comparison

2. For SEMANTIC COMPARISON, use these MANDATORY RULES:
   - Judge "True" if the generated answer matches the SEMANTIC MEANING of EITHER the expected answer OR ANY of the answer aliases
   - Judge "True" WHENEVER the general meaning or core concept is the same as either the expected answer or any alias
   - Judge "True" if one answer is GENERAL and one is SPECIFIC about the same thing
   - Judge "True" if one answer names a CATEGORY (e.g., "missionaries") and the other provides SPECIFIC INSTANCES of that category (e.g., "Augustine was sent by Pope Gregory")
   - Judge "True" if one answer gives a BRIEF fact and the other ELABORATES with more details
   - Judge "True" if one answer is more detailed but does NOT contradict the other
   - Judge "False" ONLY if the answers directly CONTRADICT all of the expected answer and all aliases, or discuss ENTIRELY different topics

3. EXTREMELY IMPORTANT RULES ABOUT SPECIFICITY:
   - When one answer is general and one is specific → TRUE
   - When one uses a category term and one gives examples → TRUE
   - When one gives "who/what" and the other adds "when/where/how/why" → TRUE
   - When one gives a person's role and the other gives their name → TRUE
   - When one refers to a group and the other names individuals → TRUE

4. Always check if the specific answer is an INSTANCE or EXAMPLE of the general answer
   - If it is, the judgment MUST be TRUE regardless of how detailed the specific answer is

5. The query is provided ONLY for context - do NOT use it in your judgment

6. IMPORTANT: The generated answer should be considered TRUE if it matches EITHER the expected answer OR ANY of the answer aliases in meaning

FINAL CHECK BEFORE SUBMITTING:
- If the generated answer could reasonably be considered matching ANY of the expected answer or aliases → TRUE
- If after reading all answers, they feel like they're talking about the same basic concept → TRUE
- If you think "the generated answer is not contradicting the expected answer or any of its aliases" → TRUE

Your response MUST follow this format:
{
  "judgment": true/false,
  "explanation": "One clear sentence explaining why the answers are compatible or contradictory."
}"""


def render_judge(question, completion):
    aliases = question["answer_aliases"]
    # Each alias wrapped in newlines and the block closed with a blank line: that is what
    # the original jinja template emits, with trim_blocks off.
    block = (
        ("\nAnswer Aliases (Additional Correct Answers):\n" + "".join(f"\n- {alias}\n" for alias in aliases) + "\n\n")
        if aliases
        else "\n"
    )
    return (
        JUDGE
        .replace("{{query}}", question["question"])
        .replace("{{expected_answer}}", question["short_answer"])
        .replace("{{aliases_block}}", block)
        .replace("{{generated_answer}}", completion.choices[0].message.content or "")
    )


print(render_judge(ok[0][0], ok[0][1])[:600])

In [ ]:
def judge(pair):
    """The judge's reply, kept whole: the verdict is derived from it, not instead of it."""
    question, completion = pair
    try:
        return client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": render_judge(question, completion)}],
            temperature=0,
            max_completion_tokens=200,
        )
    except OpenAIError as error:
        # Every failure this call can produce -- transport, timeout, rate limit, a
        # rejected request -- is an OpenAIError, and one of them should not cost the
        # run. Anything else is a bug in the code above and should not be caught here.
        return error


def read_judgment(content):
    """The judge's verdict as a bool, or None when the reply cannot be read as one.

    A bare `json.loads` is not enough. Models wrap the object in prose or in a ```json
    fence often enough to matter, so an unparsed reply falls back to scanning for the
    token. And only a real boolean counts: `bool("false")` is True, so a judge that emits
    the value as a string -- which a loose schema invites -- would otherwise mark every
    wrong answer correct, silently.
    """
    with contextlib.suppress(json.JSONDecodeError, KeyError, TypeError):
        verdict = json.loads(content)["judgment"]
        if isinstance(verdict, bool):
            return verdict

    lowered = (content if isinstance(content, str) else "").lower()
    if '"judgment": true' in lowered:
        return True
    if '"judgment": false' in lowered:
        return False
    return None


def hallucinated(verdict):
    """`judgment: true` means the answer was CORRECT, so the flag is its negation.

    The one place the judge's convention and the detector's meet. None when the reply
    cannot be read as a verdict at all.
    """
    if isinstance(verdict, Exception):
        return None
    judgment = read_judgment(verdict.choices[0].message.content)
    return None if judgment is None else not judgment


with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    verdicts = list(pool.map(judge, ok))

# The judge's replies, in the Batch output shape -- the file the CLI and the training
# notebook both read. The verdict stays as the judge wrote it; nothing is distilled out.
with JUDGMENTS.open("w", encoding="utf-8") as out:
    for (question, _), verdict in zip(ok, verdicts):
        failed = isinstance(verdict, Exception)
        out.write(
            json.dumps(
                {
                    "id": f"chatcmpl-judge-{question['question_id']}",
                    "custom_id": question["question_id"],
                    "response": None if failed else {"status_code": 200, "body": verdict.model_dump()},
                    "error": {"message": type(verdict).__name__} if failed else None,
                },
                # ASCII-escaped, unlike questions.json: every reader of these files splits
                # them with `splitlines()`, which breaks on U+2028, U+2029 and U+0085 --
                # characters JSON does not require escaping and a model can emit.
                ensure_ascii=True,
            )
            + "\n"
        )

# `judgment: true` means the answer was CORRECT, so the label is its negation. The one
# place the judge's convention and the detector's meet.
labels = [(pair, hallucinated(v)) for pair, v in zip(ok, verdicts)]
judgments = [(question, completion, flag) for (question, completion), flag in labels if flag is not None]

dropped = len(verdicts) - len(judgments)
print(f"wrote {JUDGMENTS}, {len(verdicts)} replies" + (f", {dropped} unreadable and dropped" if dropped else ""))

# A reply cut off at `max_completion_tokens` is invalid JSON, so it lands in `dropped` with
# nothing saying why. The judge answers in JSON *and* explains itself, so this is the cap
# that runs out first.
truncated = sum(1 for v in verdicts if not isinstance(v, Exception) and v.choices[0].finish_reason == "length")
if truncated:
    print(f"  {truncated} reply(ies) hit max_completion_tokens; raise it and rerun step 3")
for question, completion, flag in judgments[:2]:
    said = completion.choices[0].message.content.strip()
    print(f"  [{'hallucination' if flag else 'grounded'}] said {said[:40]!r} (gold: {question['short_answer']!r})")

In [ ]:
import numpy as np

y = np.array([flag for _, _, flag in judgments], dtype=int)

# Both classes are needed, and this is where a run fails cheaply rather than inside `fit`.
# All-correct means the questions were too easy for this model; all-wrong usually means it
# is not answering in the short form the judge expects.
assert 0 < y.sum() < len(y), f"labels are single-class ({y.sum()}/{len(y)}); the questions need to be harder or easier"

print(f"{len(y)} judged answers, {y.sum()} hallucinations ({y.mean():.0%})")

## Step 4 — fit and evaluate

The three files are written, so from here everything is repeatable for free. This step
reads them back rather than using what is still in memory -- which is what makes it true
that you can come back tomorrow, change `k`, and refit without paying for generation
again.

`trainable=True` returns an unfitted pipeline -- parser, entropy reduction, logistic
regression -- that takes the raw responses, so there is no feature extraction to write.

**ROC-AUC** scores the ranking, which governs triage by score and is what the paper
reports; the **classification report** scores the decisions at 0.5, where recall on the
`hallucination` row is the fraction actually flagged. Only the AUC carries over to another
threshold.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

from artefactual.scoring import wepr


# Read back from disk, not from the variables above: this is the path a fresh kernel takes,
# and the one anything else reading these files takes too.
def completions(path):
    """Every usable completion in a Batch output file, by `custom_id`.

    `.get` and the blank-line skip because this reads a file, not the variables above: it
    is the same code that would read a file written by anything else.
    """
    found = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        envelope = row.get("response") or {}
        if row.get("error") is not None or not envelope:
            continue
        found[row["custom_id"]] = envelope.get("body", envelope)
    return found


answers = completions(RESPONSES)
labelled, unreadable = [], 0
for custom_id, completion in completions(JUDGMENTS).items():
    if custom_id not in answers:
        continue
    # The same reader step 3 used. A fenced or prose-wrapped reply is the common case, and
    # a bare `json.loads` here would end the run on the last cell that does anything.
    judgment = read_judgment(completion["choices"][0]["message"]["content"])
    if judgment is None:
        unreadable += 1
        continue
    # `judgment: true` means the answer was correct, so the label is its negation.
    labelled.append((answers[custom_id], int(not judgment)))

if unreadable:
    print(f"{unreadable} verdict(s) could not be read and are not trained on")

responses = [completion for completion, _ in labelled]
y = np.array([label for _, label in labelled])
print(f"read {len(responses)} labelled answers back from {RESPONSES.name} and {JUDGMENTS.name}")

# Enough of the rarer class to sit on both sides of the split and mean something once
# there. At N_QUESTIONS = 25 a capable model can land under this; that is the smoke test
# telling you it was a smoke test.
rarer = min(y.sum(), len(y) - y.sum())
assert rarer >= 5, (
    f"only {rarer} answer(s) in the rarer class out of {len(y)}; a holdout cannot say "
    f"anything at that size. Raise N_QUESTIONS and rerun steps 1-3."
)

x_train, x_test, y_train, y_test = train_test_split(responses, y, test_size=0.25, stratify=y, random_state=SEED)
detector = wepr(k=K, trainable=True).fit(x_train, y_train)
print(f"fitted on {len(y_train)}, holding out {len(y_test)}")

scores = detector.predict_proba(x_test)[:, 1]
print(f"\nROC-AUC: {roc_auc_score(y_test, scores):.2f}")
print(classification_report(y_test, scores >= 0.5, target_names=["grounded", "hallucination"], zero_division=0))

In [ ]:
path = detector.save_estimator("wepr-generated.skops")
print(f"wrote {path}")

reloaded = wepr(path, k=K)

# Held-out answers: the rows the fit above never saw. `responses` holds the completions as
# they were read back from the file, so they are plain dicts rather than SDK objects.
for completion, label in list(zip(x_test, y_test))[:5]:
    probability = reloaded.predict_proba(completion)[0, 1]
    said = completion["choices"][0]["message"]["content"].strip()
    print(f"[{'hallucination' if label else 'grounded    '}] P={probability:.3f}  said {said[:40]!r}")

## Where to go next

- **Refit without regenerating.** The three files are the expensive part. Point
  [train_wepr](train_wepr.ipynb) at them to try a different `k`, `epr` instead of `wepr`, or
  relabelled verdicts — none of it costs another request. That notebook opens the three
  `*_sample` files it ships with, so change the three `Path(...)` constants at the top of
  it to the names written here.
- **Feed the CLI instead.** This run's two Batch files go straight into the
  `train_detector.py` script in
  [the repository](https://github.com/artefactory/artefactual/blob/main/scripts/train_detector.py),
  which also reports the bootstrap confidence intervals the fit above does not — worth
  having, because a holdout this size cannot pin a score down on its own.
- **More questions.** The generation is the cost and the fit is seconds, so raise
  `N_QUESTIONS` rather than economising on labels.
- **At thousands of questions**, stop making one request per answer. Batch submission
  takes a JSONL of requests and returns the JSONL these files already are, at roughly half
  the price: OpenAI's [Batch API](https://platform.openai.com/docs/api-reference/batch) if
  your provider hosts one, or an offline batch runner against a self-hosted server. Only
  steps 2 and 3 change -- what they write, and everything after it, stays as it is.
- **Another dataset.** Only step 1 changes. Training data should resemble the traffic being
  scored: the paper's numbers drop 10-20 points when a TriviaQA-trained detector meets
  WebQuestions.